## Example 1: Absolute URIs with the default URI rule

In [ ]:
import rfc3986_validator

# Example 1: Absolute URIs with the default URI rule
print("Example 1: Absolute URIs with the default URI rule")

try:
    cases = {
        'http://example.com/a?b#c': True,
        'https://user:pw@[::1]:8080/p': True,   # userinfo, IPv6 literal and port
        'urn:isbn:0451450523': True,
        'mailto:a@example.com': True,
        'http://exa mple.com': False,           # a space is not allowed anywhere
        'http://example.com/%zz': False,        # '%' must be followed by two hex digits
        '//example.com/p': False,               # no scheme: a relative reference, not a URI
    }
    for uri, expected in cases.items():
        result = rfc3986_validator.validate_rfc3986(uri, rule='URI')
        print(f"  {uri!r:34} -> {'valid' if result else 'invalid'}")
        assert bool(result) is expected, (uri, result)

    print("rfc3986_validator example 1 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"rfc3986_validator example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 finished")

## Example 2: URI references and relative references

In [ ]:
import rfc3986_validator

# Example 2: URI references and relative references
print("Example 2: URI references and relative references")

try:
    # 'URI_reference' accepts a full URI or a relative reference (RFC 3986 section 4.1)
    relative_refs = ['//example.com/p', '/a/b?c', 'a/b', '#frag', '?q=1', '']
    for ref in relative_refs:
        as_uri = rfc3986_validator.validate_rfc3986(ref, rule='URI')
        as_ref = rfc3986_validator.validate_rfc3986(ref, rule='URI_reference')
        print(f"  {ref!r:20} URI={bool(as_uri)!s:5} URI_reference={bool(as_ref)}")
        assert as_uri is None and as_ref is not None, ref

    # absolute URIs are URI references too, and invalid text is invalid under both rules
    assert rfc3986_validator.validate_rfc3986('http://example.com/a?b#c', rule='URI_reference')
    assert not rfc3986_validator.validate_rfc3986('http://exa mple.com', rule='URI_reference')

    print("rfc3986_validator example 2 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"rfc3986_validator example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 finished")

## Example 3: Return values and rule names

In [ ]:
import rfc3986_validator

# Example 3: Return values and rule names
print("Example 3: Return values and rule names")

try:
    import re

    # A valid string returns the regex match object spanning the whole input; an invalid one returns None
    match = rfc3986_validator.validate_rfc3986('http://example.com/a?b#c')
    print(f"  match type: {type(match).__name__}, matched: {match.group(0)!r}")
    assert isinstance(match, re.Match) and match.group(0) == 'http://example.com/a?b#c', match
    assert rfc3986_validator.validate_rfc3986('http://exa mple.com') is None

    # The default rule is 'URI', so a relative path is rejected unless URI_reference is asked for
    assert rfc3986_validator.validate_rfc3986('/a/b') is None
    assert rfc3986_validator.validate_rfc3986('/a/b', rule='URI_reference') is not None

    # Only 'URI' and 'URI_reference' are supported; any other rule name raises ValueError
    try:
        rfc3986_validator.validate_rfc3986('/a/b', rule='relative_ref')
    except ValueError as err:
        print(f"  unknown rule rejected: {err}")
    else:
        raise AssertionError("an unknown rule name was accepted")

    print("rfc3986_validator example 3 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"rfc3986_validator example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 finished")

## Example 4: Hosts, ports and percent-encoding

In [ ]:
import rfc3986_validator

# Example 4: Hosts, ports and percent-encoding
print("Example 4: Hosts, ports and percent-encoding")

try:
    cases = {
        'http://192.168.0.1:80/': True,             # IPv4 host with port
        'http://[2001:db8::1]/': True,              # IPv6 literal
        'http://[v1.fe80::a+en1]/': True,           # IPvFuture literal
        'http://example.com:/': True,               # an empty port is allowed by the grammar
        'http://example.com/%E2%82%AC': True,       # percent-encoded UTF-8 (the euro sign)
        'http://example.com/%e2%82%ac': True,       # lower-case hex digits are allowed
        'http://例え.jp/': False,                   # raw non-ASCII needs an IRI, not a URI
        'http://example.com/a b': False,            # space in the path
        'ht tp://example.com/': False,              # space in the scheme
    }
    for uri, expected in cases.items():
        result = rfc3986_validator.validate_rfc3986(uri)
        print(f"  {uri!r:32} -> {'valid' if result else 'invalid'}")
        assert bool(result) is expected, (uri, result)

    print("rfc3986_validator example 4 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"rfc3986_validator example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 finished")

## Example 5: Filtering a list of links

In [ ]:
import rfc3986_validator

# Example 5: Filtering a list of links
print("Example 5: Filtering a list of links")

try:
    links = [
        'https://example.com/docs/index.html',
        'ftp://ftp.example.com/pub/file.txt',
        'javascript:void(0)',
        'https://example.com/search?q=a b',
        '../relative/path',
        'https://example.com/%GG',
    ]
    absolute = [u for u in links if rfc3986_validator.validate_rfc3986(u, rule='URI')]
    references = [u for u in links if rfc3986_validator.validate_rfc3986(u, rule='URI_reference')]
    print(f"  absolute URIs: {absolute}")
    print(f"  URI references: {references}")
    # 'javascript:void(0)' is syntactically a URI; the grammar does not judge whether a scheme is safe
    assert absolute == ['https://example.com/docs/index.html', 'ftp://ftp.example.com/pub/file.txt', 'javascript:void(0)'], absolute
    assert references == absolute + ['../relative/path'], references

    print("rfc3986_validator example 5 completed successfully")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"rfc3986_validator example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 finished")